# 01 - Custom Chatbot / Assistant Fine-Tuning

This notebook is the **entry point** for local fine-tuning using a subset of OASST1, Alpaca, and Dolly datasets with QLoRA.

Base assumptions for this machine:
- GPU: RTX 2060 (6GB VRAM)
- Local fast training with dataset subsets
- Stack: `transformers`, `datasets`, `accelerate`, `peft`, `FastAPI`, `uvicorn`


In [ ]:
# If needed, install dependencies once
# !pip install -U pip
# !pip install -e .


In [ ]:
# Confirm GPU
!nvidia-smi


In [ ]:
from pathlib import Path
import yaml

config = {
    'project_name': 'custom_chatbot_assistant',
    'seed': 42,
    'model': {
        'base_model_name': 'EleutherAI/gpt-neo-125M',
        'max_seq_length': 512,
        'load_in_4bit': False,
        'torch_dtype': 'float16',
        'lora_r': 16,
        'lora_alpha': 32,
        'lora_dropout': 0.05,
        'target_modules': ['q_proj', 'v_proj']
    },
    'data': {
        'cache_dir': 'data/cache',
        'processed_dir': 'data/processed',
        'train_split': 0.9,
        'val_split': 0.05,
        'test_split': 0.05,
        'max_samples_total': 12000,
        'max_samples_per_dataset': {
            'oasst1': 6000,
            'alpaca_cleaned': 3000,
            'dolly_15k': 3000
        },
        'min_prompt_chars': 8,
        'min_response_chars': 8
    },
    'training': {
        'output_dir': 'models/custom_chatbot',
        'per_device_train_batch_size': 4,
        'per_device_eval_batch_size': 2,
        'gradient_accumulation_steps': 8,
        'num_train_epochs': 1,
        'learning_rate': 2e-4,
        'weight_decay': 0.01,
        'warmup_ratio': 0.03,
        'logging_steps': 10,
        'save_strategy': 'epoch',
        'evaluation_strategy': 'epoch',
        'gradient_checkpointing': True,
        'fp16': True,
        'max_grad_norm': 0.3
    },
    'generation': {
        'max_new_tokens': 180,
        'temperature': 0.7,
        'top_p': 0.9,
        'repetition_penalty': 1.05
    }
}

config_path = Path('../configs/train_config.yaml').resolve()
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print(f'Config written: {config_path}')


In [ ]:
# Prepare dataset (dedupe, formatting, tokenization-ready)
!python ../scripts/prepare_data.py --config configs/train_config.yaml


In [ ]:
# Train with accelerate-backed Trainer
!accelerate launch ../scripts/train.py --config configs/train_config.yaml


In [ ]:
# Evaluate BLEU / ROUGE / exact-match accuracy on validation/test subsets
!python ../scripts/evaluate.py --config configs/train_config.yaml --split test --max_eval_samples 300


In [ ]:
# Serve inference API
# !uvicorn ../deploy/app:app --host 0.0.0.0 --port 8000 --reload
